In [1]:
import pandas as pd
import numpy as np

movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

In [2]:
unique_genres = (
    movies["genres"]
    .str.split("|")
    .explode()
    .unique()
)

available_genres = sorted([
    genre for genre in unique_genres
    if genre not in ["(no genres listed)", "IMAX"]
])

print("=== Available Genres ===")
print(available_genres)

print("Number of genres:", len(available_genres))

=== Available Genres ===
['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
Number of genres: 18


In [3]:
genre_matrix = movies["genres"].str.get_dummies(
    sep="|"
)

print("=== Genre Matrix ===")
print(genre_matrix.head())

print("\nMatrix shape:", genre_matrix.shape)

=== Genre Matrix ===
   (no genres listed)  Action  Adventure  Animation  Children  Comedy  Crime  \
0                   0       0          1          1         1       1      0   
1                   0       0          1          0         1       0      0   
2                   0       0          0          0         0       1      0   
3                   0       0          0          0         0       1      0   
4                   0       0          0          0         0       1      0   

   Documentary  Drama  Fantasy  Film-Noir  Horror  IMAX  Musical  Mystery  \
0            0      0        1          0       0     0        0        0   
1            0      0        1          0       0     0        0        0   
2            0      0        0          0       0     0        0        0   
3            0      1        0          0       0     0        0        0   
4            0      0        0          0       0     0        0        0   

   Romance  Sci-Fi  Thriller  War  

In [4]:
genre_matrix = genre_matrix[available_genres]

print("=== Filtered Genre Matrix ===")
print(genre_matrix.head())

print("\nMatrix shape:", genre_matrix.shape)

=== Filtered Genre Matrix ===
   Action  Adventure  Animation  Children  Comedy  Crime  Documentary  Drama  \
0       0          1          1         1       1      0            0      0   
1       0          1          0         1       0      0            0      0   
2       0          0          0         0       1      0            0      0   
3       0          0          0         0       1      0            0      1   
4       0          0          0         0       1      0            0      0   

   Fantasy  Film-Noir  Horror  Musical  Mystery  Romance  Sci-Fi  Thriller  \
0        1          0       0        0        0        0       0         0   
1        1          0       0        0        0        0       0         0   
2        0          0       0        0        0        1       0         0   
3        0          0       0        0        0        1       0         0   
4        0          0       0        0        0        0       0         0   

   War  Western  
0 

In [5]:
genre_matrix.index = movies["movieId"]

print("=== Genre Matrix with Movie IDs ===")
print(genre_matrix.head())

=== Genre Matrix with Movie IDs ===
         Action  Adventure  Animation  Children  Comedy  Crime  Documentary  \
movieId                                                                       
1             0          1          1         1       1      0            0   
2             0          1          0         1       0      0            0   
3             0          0          0         0       1      0            0   
4             0          0          0         0       1      0            0   
5             0          0          0         0       1      0            0   

         Drama  Fantasy  Film-Noir  Horror  Musical  Mystery  Romance  Sci-Fi  \
movieId                                                                         
1            0        1          0       0        0        0        0       0   
2            0        1          0       0        0        0        0       0   
3            0        0          0       0        0        0        1       0   
4    

In [6]:
target_movie_id = 1

print("=== Movie Information ===")

print(
    movies.loc[
        movies["movieId"] == target_movie_id,
        ["title", "genres"]
    ]
)

print("\n=== Movie Genre Vector ===")

print(genre_matrix.loc[target_movie_id])

=== Movie Information ===
              title                                       genres
0  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy

=== Movie Genre Vector ===
Action         0
Adventure      1
Animation      1
Children       1
Comedy         1
Crime          0
Documentary    0
Drama          0
Fantasy        1
Film-Noir      0
Horror         0
Musical        0
Mystery        0
Romance        0
Sci-Fi         0
Thriller       0
War            0
Western        0
Name: 1, dtype: int64


In [7]:
genre_count_per_movie = genre_matrix.sum(axis=1)

zero_genre_movies = genre_matrix[
    genre_count_per_movie == 0
]

print("=== Movies with No Valid Genres ===")

print("Number of movies:", len(zero_genre_movies))

print(zero_genre_movies.head())

=== Movies with No Valid Genres ===
Number of movies: 34
         Action  Adventure  Animation  Children  Comedy  Crime  Documentary  \
movieId                                                                       
114335        0          0          0         0       0      0            0   
122888        0          0          0         0       0      0            0   
122896        0          0          0         0       0      0            0   
129250        0          0          0         0       0      0            0   
132084        0          0          0         0       0      0            0   

         Drama  Fantasy  Film-Noir  Horror  Musical  Mystery  Romance  Sci-Fi  \
movieId                                                                         
114335       0        0          0       0        0        0        0       0   
122888       0        0          0       0        0        0        0       0   
122896       0        0          0       0        0        0     

In [8]:
print("=== Genre Matrix Validation ===")

print("Number of movies:", genre_matrix.shape[0])

print("Number of features:", genre_matrix.shape[1])

print("Missing values:",
      genre_matrix.isnull().sum().sum())

print("Duplicate movie IDs:",
      genre_matrix.index.duplicated().sum())

print("Unique values:",
      np.unique(genre_matrix.to_numpy()))

print("\n=== Genres per Movie Statistics ===")
print(genre_count_per_movie.describe())

=== Genre Matrix Validation ===
Number of movies: 9742
Number of features: 18
Missing values: 0
Duplicate movie IDs: 0
Unique values: [0 1]

=== Genres per Movie Statistics ===
count    9742.000000
mean        2.247177
std         1.108284
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        10.000000
dtype: float64


In [9]:
movie_stats = ratings.groupby("movieId")["rating"].agg(
    ["count", "mean"]
)

movie_stats = movie_stats.rename(
    columns={
        "count": "rating_count",
        "mean": "rating_mean"
    }
)

movie_stats = movie_stats.reset_index()

print("=== Movie Rating Statistics ===")
print(movie_stats.head(10))

=== Movie Rating Statistics ===
   movieId  rating_count  rating_mean
0        1           215     3.920930
1        2           110     3.431818
2        3            52     3.259615
3        4             7     2.357143
4        5            49     3.071429
5        6           102     3.946078
6        7            54     3.185185
7        8             8     2.875000
8        9            16     3.125000
9       10           132     3.496212


In [10]:
movies_prepared = movies.merge(
    movie_stats,
    on="movieId",
    how="left",
    validate="one_to_one"
)

print("=== Prepared Movie Data ===")
print(movies_prepared.head())

print("\nShape:", movies_prepared.shape)

=== Prepared Movie Data ===
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  rating_count  rating_mean  
0  Adventure|Animation|Children|Comedy|Fantasy         215.0     3.920930  
1                   Adventure|Children|Fantasy         110.0     3.431818  
2                               Comedy|Romance          52.0     3.259615  
3                         Comedy|Drama|Romance           7.0     2.357143  
4                                       Comedy          49.0     3.071429  

Shape: (9742, 5)


In [11]:
global_mean = ratings["rating"].mean()

movies_prepared["rating_count"] = (
    movies_prepared["rating_count"].fillna(0)
)

movies_prepared["rating_mean"] = (
    movies_prepared["rating_mean"].fillna(global_mean)
)

print("Global Mean Rating:", global_mean)

print("\nMissing Values:")
print(
    movies_prepared[
        ["rating_count", "rating_mean"]
    ].isnull().sum()
)

Global Mean Rating: 3.501556983616962

Missing Values:
rating_count    0
rating_mean     0
dtype: int64


In [12]:
m = 10

v = movies_prepared["rating_count"]
R = movies_prepared["rating_mean"]
C = global_mean

movies_prepared["weighted_rating"] = (
    (v / (v + m)) * R
    + (m / (v + m)) * C
)

print("=== Weighted Ratings ===")

print(
    movies_prepared[
        ["title", "rating_count", "rating_mean", "weighted_rating"]
    ].head(10)
)

=== Weighted Ratings ===
                                title  rating_count  rating_mean  \
0                    Toy Story (1995)         215.0     3.920930   
1                      Jumanji (1995)         110.0     3.431818   
2             Grumpier Old Men (1995)          52.0     3.259615   
3            Waiting to Exhale (1995)           7.0     2.357143   
4  Father of the Bride Part II (1995)          49.0     3.071429   
5                         Heat (1995)         102.0     3.946078   
6                      Sabrina (1995)          54.0     3.185185   
7                 Tom and Huck (1995)           8.0     2.875000   
8                 Sudden Death (1995)          16.0     3.125000   
9                    GoldenEye (1995)         132.0     3.496212   

   weighted_rating  
0         3.902291  
1         3.437630  
2         3.298638  
3         3.030328  
4         3.144332  
5         3.906389  
6         3.234618  
7         3.223087  
8         3.269830  
9         3.4965

In [13]:
top_weighted_movies = movies_prepared.sort_values(
    by="weighted_rating",
    ascending=False
)

print("=== Top 10 Weighted Rated Movies ===")

print(
    top_weighted_movies[
        [
            "title",
            "rating_count",
            "rating_mean",
            "weighted_rating"
        ]
    ].head(10).to_string(index=False)
)

=== Top 10 Weighted Rated Movies ===
                                                                      title  rating_count  rating_mean  weighted_rating
                                           Shawshank Redemption, The (1994)         317.0     4.429022         4.400659
                                                      Godfather, The (1972)         192.0     4.289062         4.250077
                                                          Fight Club (1999)         218.0     4.272936         4.239103
                                             Godfather: Part II, The (1974)         129.0     4.259690         4.205148
                                                 Usual Suspects, The (1995)         204.0     4.237745         4.203344
                                  Star Wars: Episode IV - A New Hope (1977)         251.0     4.231076         4.203125
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)          97.0     4.268041         4.196407
   

In [14]:
movies_prepared["rating_difference"] = (
    movies_prepared["weighted_rating"]
    - movies_prepared["rating_mean"]
)

print("=== Rating Adjustment Examples ===")

print(
    movies_prepared[
        [
            "title",
            "rating_count",
            "rating_mean",
            "weighted_rating",
            "rating_difference"
        ]
    ].sort_values(
        by="rating_count",
        ascending=True
    ).head(15).to_string(index=False)
)

=== Rating Adjustment Examples ===
                                       title  rating_count  rating_mean  weighted_rating  rating_difference
                          Chalet Girl (2011)           0.0     3.501557         3.501557                0.0
                Browning Version, The (1951)           0.0     3.501557         3.501557                0.0
                   Call Northside 777 (1948)           0.0     3.501557         3.501557                0.0
                    Twentieth Century (1934)           0.0     3.501557         3.501557                0.0
          In the Realms of the Unreal (2004)           0.0     3.501557         3.501557                0.0
                 Mutiny on the Bounty (1962)           0.0     3.501557         3.501557                0.0
                Roaring Twenties, The (1939)           0.0     3.501557         3.501557                0.0
                    This Gun for Hire (1942)           0.0     3.501557         3.501557             

In [15]:
print("=== Weighted Rating Validation ===")

print("Number of movies:", len(movies_prepared))

print(
    "Missing weighted ratings:",
    movies_prepared["weighted_rating"].isnull().sum()
)

print(
    "Minimum weighted rating:",
    movies_prepared["weighted_rating"].min()
)

print(
    "Maximum weighted rating:",
    movies_prepared["weighted_rating"].max()
)

print(
    "Movies without ratings:",
    (movies_prepared["rating_count"] == 0).sum()
)

=== Weighted Rating Validation ===
Number of movies: 9742
Missing weighted ratings: 0
Minimum weighted rating: 2.259157580557573
Maximum weighted rating: 4.400659234972996
Movies without ratings: 18


In [16]:
min_rating = movies_prepared["weighted_rating"].min()
max_rating = movies_prepared["weighted_rating"].max()

movies_prepared["normalized_rating"] = (
    (movies_prepared["weighted_rating"] - min_rating)
    / (max_rating - min_rating)
)

print("=== Normalized Weighted Ratings ===")

print(
    movies_prepared[
        [
            "title",
            "weighted_rating",
            "normalized_rating"
        ]
    ].head(10).to_string(index=False)
)

=== Normalized Weighted Ratings ===
                             title  weighted_rating  normalized_rating
                  Toy Story (1995)         3.902291           0.767281
                    Jumanji (1995)         3.437630           0.550302
           Grumpier Old Men (1995)         3.298638           0.485398
          Waiting to Exhale (1995)         3.030328           0.360107
Father of the Bride Part II (1995)         3.144332           0.413343
                       Heat (1995)         3.906389           0.769195
                    Sabrina (1995)         3.234618           0.455503
               Tom and Huck (1995)         3.223087           0.450119
               Sudden Death (1995)         3.269830           0.471945
                  GoldenEye (1995)         3.496589           0.577833


In [17]:
print("=== Normalization Validation ===")

print(
    "Minimum normalized rating:",
    movies_prepared["normalized_rating"].min()
)

print(
    "Maximum normalized rating:",
    movies_prepared["normalized_rating"].max()
)

print(
    "Missing values:",
    movies_prepared["normalized_rating"].isnull().sum()
)

=== Normalization Validation ===
Minimum normalized rating: 0.0
Maximum normalized rating: 1.0
Missing values: 0


In [18]:
print("=== Before and After Normalization ===")

comparison = movies_prepared[
    [
        "title",
        "rating_count",
        "weighted_rating",
        "normalized_rating"
    ]
].sort_values(
    by="weighted_rating",
    ascending=False
)

print(comparison.head(10).to_string(index=False))

=== Before and After Normalization ===
                                                                      title  rating_count  weighted_rating  normalized_rating
                                           Shawshank Redemption, The (1994)         317.0         4.400659           1.000000
                                                      Godfather, The (1972)         192.0         4.250077           0.929684
                                                          Fight Club (1999)         218.0         4.239103           0.924560
                                             Godfather: Part II, The (1974)         129.0         4.205148           0.908704
                                                 Usual Suspects, The (1995)         204.0         4.203344           0.907861
                                  Star Wars: Episode IV - A New Hope (1977)         251.0         4.203125           0.907759
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)    

In [19]:
movie_features = movies_prepared.merge(
    genre_matrix.reset_index(),
    on="movieId",
    how="left",
    validate="one_to_one"
)

print("=== Final Movie Features ===")
print(movie_features.head())

print("\nShape:", movie_features.shape)

=== Final Movie Features ===
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  rating_count  rating_mean  \
0  Adventure|Animation|Children|Comedy|Fantasy         215.0     3.920930   
1                   Adventure|Children|Fantasy         110.0     3.431818   
2                               Comedy|Romance          52.0     3.259615   
3                         Comedy|Drama|Romance           7.0     2.357143   
4                                       Comedy          49.0     3.071429   

   weighted_rating  rating_difference  normalized_rating  Action  Adventure  \
0         3.902291          -0.018639           0.767281       0          1   
1         3.437630           0.005812           0

In [20]:
print("=== Final Data Validation ===")

print("Shape:", movie_features.shape)

print(
    "Duplicate movie IDs:",
    movie_features["movieId"].duplicated().sum()
)

print(
    "Missing values:",
    movie_features.isnull().sum().sum()
)

print(
    "Number of genre features:",
    len(available_genres)
)

print(
    "Movies with no valid genres:",
    (movie_features[available_genres].sum(axis=1) == 0).sum()
)

=== Final Data Validation ===
Shape: (9742, 26)
Duplicate movie IDs: 0
Missing values: 0
Number of genre features: 18
Movies with no valid genres: 34


In [21]:
content_candidates = movie_features[
    movie_features[available_genres].sum(axis=1) > 0
].copy()

print("=== Content-Based Candidates ===")

print("Total movies:", len(movie_features))
print("Content candidates:", len(content_candidates))
print(
    "Excluded from genre-based recommendation:",
    len(movie_features) - len(content_candidates)
)

=== Content-Based Candidates ===
Total movies: 9742
Content candidates: 9708
Excluded from genre-based recommendation: 34


In [22]:
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

movie_features.to_csv(
    processed_dir / "movie_features.csv",
    index=False
)

content_candidates.to_csv(
    processed_dir / "content_candidates.csv",
    index=False
)

print("Processed data saved successfully.")

Processed data saved successfully.
